# Parameter-Efficient Fine-Tuning (PEFT): LoRA and Quantization — Colab

This notebook mirrors `docs/chapter_llm_ft/peft.md`. You will:
- Understand 8-bit/4-bit quantization and why it helps
- Apply LoRA adapters to reduce trainable parameters
- Fine-tune a small chat model with `trl.SFTTrainer`
- Optionally merge adapters for simple deployment

We default to a small public model for Colab (TinyLlama). If you have more VRAM, you can switch to Llama 3 8B.


In [ ]:
# Install libraries for PEFT demos
import sys, subprocess

def pip_install(packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *packages])

pip_install([
    "transformers",
    "datasets",
    "accelerate",
    "trl",
    "peft",
    "bitsandbytes",
])

import torch, transformers, datasets, accelerate, trl, peft
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)


## Quantization (8-bit vs 4-bit) and loading a small model

We’ll use TinyLlama by default; swap to Llama 3 8B if you have VRAM. 4-bit (nf4) is the most memory-efficient and works well with LoRA (QLoRA).


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

small_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

bnb_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(small_model_id)
model = AutoModelForCausalLM.from_pretrained(
    small_model_id,
    device_map="auto",
    quantization_config=bnb_4bit,
    torch_dtype=compute_dtype,
)

print("Loaded:", small_model_id)


## Add LoRA adapters

LoRA trains small adapters on top of frozen base weights. This is cheap and effective.

We’ll use `target_modules='all-linear'` as a robust default.


In [ ]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
# Align adapter dtype with desired compute dtype (float16 on T4)
try:
    model = model.to(compute_dtype)
except Exception:
    pass
model.print_trainable_parameters()


## SFT with a tiny messages dataset

We create a small JSONL dataset and fine-tune briefly using `trl.SFTTrainer`. This mirrors the PEFT lecture examples but sized for Colab.


In [ ]:
import json
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer, setup_chat_format

records = [
    {
        "messages": [
            {"role": "system", "content": "You are a clinical calculator assistant."},
            {"role": "user", "content": "Patient Note: BMI example.\nQuestion: Height 1.75m, Weight 70kg.\nAnswer:"},
            {"role": "assistant", "content": "22.86"}
        ]
    },
]

with open("train_dataset.json", "w") as f:
    for r in records:
        f.write(json.dumps(r) + "\n")

# Ensure chat formatting only if missing; also ensure pad token
if getattr(tokenizer, "chat_template", None) is None:
    model, tokenizer = setup_chat_format(model, tokenizer)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

train_ds = load_dataset("json", data_files="train_dataset.json", split="train")

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
args = TrainingArguments(
    output_dir="tinyllama-peft-sft",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    bf16=use_bf16,
    fp16=not use_bf16,
    tf32=False,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
)

import inspect

sft_kwargs = dict(
    model=model,
    train_dataset=train_ds,
    args=args,
    max_seq_length=1024,
    packing=True,
    dataset_kwargs={"add_special_tokens": False, "append_concat_token": False},
)
params = inspect.signature(SFTTrainer.__init__).parameters
if "tokenizer" in params:
    sft_kwargs["tokenizer"] = tokenizer
elif "processing_class" in params:
    sft_kwargs["processing_class"] = tokenizer

trainer = SFTTrainer(**sft_kwargs)

trainer.train()
trainer.save_model()


## Merge adapters (optional)

If you want a single checkpoint without PEFT at inference time, you can merge adapters on CPU.


In [ ]:
from peft import AutoPeftModelForCausalLM

peft_dir = "tinyllama-peft-sft"
try:
    peft_model = AutoPeftModelForCausalLM.from_pretrained(
        peft_dir, torch_dtype=torch.float16, low_cpu_mem_usage=True
    )
    merged = peft_model.merge_and_unload()
    merged.save_pretrained(peft_dir, safe_serialization=True, max_shard_size="2GB")
    print("Merged and saved into:", peft_dir)
except Exception as e:
    print("Skipping merge (likely running in a fresh session):", e)


## Prepare model for inference (disable checkpointing)

To avoid caching warnings and ensure generation runs smoothly, disable gradient checkpointing and enable cache before inference.
